In [ ]:
from meteostat import Point, Hourly
from datetime import datetime

year = 2025

start = datetime(year, 1, 1)
end = datetime(year, 12, 11)

berlin = Point(52.52, 13.405)

data = Hourly(
    berlin,
    start=start,
    end=end,
)

df_weather = data.fetch()
df_weather

,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2025-01-01 00:00:00,2.0,-0.9,81.0,0.0,0.0,226.0,22.2,40.8,1018.9,0.0,4.0
2025-01-01 01:00:00,2.3,-1.0,79.0,0.0,0.0,222.0,22.2,40.8,1018.3,0.0,4.0
2025-01-01 02:00:00,2.7,-0.7,78.0,0.0,0.0,219.0,22.2,42.6,1017.7,0.0,4.0
2025-01-01 03:00:00,3.2,-0.4,77.0,0.0,0.0,219.0,24.1,44.5,1017.0,0.0,4.0
2025-01-01 04:00:00,3.5,-0.3,76.0,0.0,0.0,221.0,24.1,44.5,1016.3,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-12-10 20:00:00,10.9,9.3,90.0,0.1,0.0,234.0,14.8,25.9,1019.0,0.0,7.0
2025-12-10 21:00:00,10.7,9.1,90.0,0.1,0.0,236.0,14.8,27.8,1019.4,0.0,7.0
2025-12-10 22:00:00,10.5,9.1,91.0,0.2,0.0,240.0,14.8,27.8,1019.5,0.0,7.0


In [ ]:
import os

out_dir = "../../data/01_raw/weather"
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/meteostat_{year}.csv"

df_weather.to_csv(out_path, index=True)

print(f"saved: {out_path}")


saved: ../../data/raw/weather/meteostat_2025.csv


# Historic Weather Forecasts

In [6]:
import requests
import pandas as pd

start_date = start.strftime("%Y-%m-%d")
end_date = end.strftime("%Y-%m-%d")

params = {
    "latitude": 52.52,
    "longitude": 13.405,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join([
        "temperature_2m",
        "wind_speed_10m",
        "shortwave_radiation",
        "cloud_cover",
    ]),
    # optional: choose model explicitly (examples: "ecmwf_ifs", "gfs", "icon")
    # "models": "ecmwf_ifs",
    "timezone": "Europe/Berlin",
}

r = requests.get(
    "https://historical-forecast-api.open-meteo.com/v1/forecast",
    params=params,
    timeout=30,
)
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])
df


,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
0,2025-01-01 00:00:00,1.0,17.9,0.0,100
1,2025-01-01 01:00:00,1.5,18.9,0.0,100
2,2025-01-01 02:00:00,1.7,20.5,0.0,100
3,2025-01-01 03:00:00,1.8,19.9,0.0,100
4,2025-01-01 04:00:00,2.3,20.6,0.0,100
...,...,...,...,...,...
8275,2025-12-11 19:00:00,8.8,11.2,0.0,35
8276,2025-12-11 20:00:00,8.2,11.2,0.0,89
8277,2025-12-11 21:00:00,7.6,10.9,0.0,90
8278,2025-12-11 22:00:00,7.2,10.2,0.0,92


In [9]:
import os

out_dir = "../../data/raw/weather"
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/open_meteo_historic_forecasts_{year}.csv"

df.to_csv(out_path, index=False)

print(f"saved: {out_path}")


saved: ../../data/raw/weather/open_meteo_historic_forecasts_2025.csv


# Quatsch

In [10]:
import requests
import pandas as pd

params = {
    "latitude": 52.52,
    "longitude": 13.405,
    "hourly": ",".join([
        "temperature_2m",
        "wind_speed_10m",
        "shortwave_radiation",
        "cloud_cover",
    ]),
    "forecast_days": 7,
    "timezone": "Europe/Berlin",
}

r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=params,
    timeout=30,
)

r.raise_for_status()
data = r.json()


In [12]:
import pandas as pd

hourly = data["hourly"]

df_weather = pd.DataFrame(hourly)
df_weather["time"] = pd.to_datetime(df_weather["time"])
df_weather["time"] = df_weather["time"].dt.tz_localize("Europe/Berlin")

df_weather

,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
0,2025-12-12 00:00:00+01:00,6.4,7.2,0.0,65
1,2025-12-12 01:00:00+01:00,6.1,8.4,0.0,100
2,2025-12-12 02:00:00+01:00,5.5,7.2,0.0,0
3,2025-12-12 03:00:00+01:00,5.2,6.5,0.0,100
4,2025-12-12 04:00:00+01:00,5.1,7.0,0.0,100
...,...,...,...,...,...
163,2025-12-18 19:00:00+01:00,7.5,11.2,0.0,93
164,2025-12-18 20:00:00+01:00,7.5,11.5,0.0,95
165,2025-12-18 21:00:00+01:00,7.4,11.1,0.0,98
166,2025-12-18 22:00:00+01:00,7.4,10.8,0.0,100
